In [1]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import loompy

In [2]:
adata = sc.read_h5ad("/rds/general/user/ao225/home/CardiaFinal/Data/Post_Processed/Science/LVCardio.h5ad")

In [3]:
gene_names_all = adata.var["feature_name"].values
is_symbol = ~pd.Series(gene_names_all).str.startswith("ENSG").values
gene_names_filtered = gene_names_all[is_symbol]
gene_positions = np.where(is_symbol)[0]

print(f"Cells: {adata.shape[0]}, Genes: {adata.shape[1]}")
print(f"Genes with proper symbols: {len(gene_names_filtered)}")
print(f"Genes without symbols (dropped): {len(gene_names_all) - len(gene_names_filtered)}")

Cells: 100528, Genes: 32383
Genes with proper symbols: 24938
Genes without symbols (dropped): 7445


In [4]:
gene_series = pd.Series(gene_names_filtered)
keep_first = ~gene_series.duplicated(keep="first")

gene_names_final = gene_series[keep_first].values
final_positions = gene_positions[keep_first.values]

print(f"After removing duplicates: {len(gene_names_final)}")
print(f"Example gene names: {gene_names_final[:10].tolist()}")

After removing duplicates: 24922
Example gene names: ['MIR1302-2HG', 'FAM138A', 'OR4F5', 'OR4F29', 'OR4F16', 'LINC01409', 'FAM87B', 'LINC00115', 'FAM41C', 'LINC02593']


In [5]:
print("Building expression matrix (genes x cells)...")

expr_cxg = adata.X[:, final_positions]  # cells x genes

# handle either sparse or dense X
if hasattr(expr_cxg, "tocsr"):
    expr_gxc = expr_cxg.T.tocsr()
else:
    expr_gxc = expr_cxg.T

cell_ids = adata.obs_names.values
print(f"Shape (genes x cells): {expr_gxc.shape}")
print(f"Type: {type(expr_gxc)}")

Building expression matrix (genes x cells)...
Shape (genes x cells): (24922, 100528)
Type: <class 'scipy.sparse._csr.csr_matrix'>


In [6]:
print(f"dtype: {expr_gxc.dtype}")

cell_info = adata.obs[["cell_states", "donor_id", "disease", "sex", "assay"]].copy()
print(cell_info.shape)
print(cell_info.head())

dtype: float32
(100528, 5)
     cell_states donor_id                 disease   sex      assay
3350      vCM1.1      DT4  dilated cardiomyopathy  male  10x 3' v3
3351        vCM4      DT4  dilated cardiomyopathy  male  10x 3' v3
3352      vCM1.0      DT4  dilated cardiomyopathy  male  10x 3' v3
3353      vCM1.1      DT4  dilated cardiomyopathy  male  10x 3' v3
3354      vCM1.3      DT4  dilated cardiomyopathy  male  10x 3' v3


In [7]:
LOOM_OUT="/rds/general/user/ao225/home/CardiaFinal/Results/SCENIC/Processed_files/lv_exp.loom"

In [8]:
dense_mtx = expr_gxc.toarray()   # do this once, keep it around

row_attrs = {"Gene": np.asarray(gene_names_final, dtype=object)}
col_attrs = {
    "CellID": np.asarray(cell_ids, dtype=object),
    "cell_states": np.asarray(cell_info["cell_states"].to_numpy(), dtype=object),
    "donor_id": np.asarray(cell_info["donor_id"].to_numpy(), dtype=object),
}

print("Writing loom...")
loompy.create(LOOM_OUT, dense_mtx, row_attrs, col_attrs)
print(f"Created: {LOOM_OUT}")

Writing loom...
Created: /rds/general/user/ao225/home/CardiaFinal/Results/SCENIC/Processed_files/lv_exp.loom
